# Visualize Exported Conversations

This notebook loads all exported conversation JSON files from a directory, shows a summary dataframe with scores, and lets you visualize one conversation by its row index.


In [ ]:
from pathlib import Path
import os

ROOT = Path(os.environ.get("REPO_ROOT", Path.cwd())).resolve()
EXPORT_DIR = Path(os.environ.get("EXPORT_DIR", "/path/to/exported_conversations")).expanduser()
assert EXPORT_DIR.exists(), f"Set EXPORT_DIR to an exported_conversations directory: {EXPORT_DIR}"


In [ ]:
import pandas as pd

def _get_score_field(record, key, default=None):
    reward = record.get("reward") or {}
    score = reward.get("score") or {}
    return score.get(key, default)

rows = []
for idx, (path, record) in enumerate(zip(export_paths, records)):
    reward = record.get("reward") or {}
    rows.append({
        "idx": idx,
        "subset": record['extra_info']['subset'],
        "file": path.name,
        "job_id": record["job"]["job_id"],
        "agent_name": record.get("agent_name"),
        "question_id": record.get("extra_info", {}).get("question_id"),
        "question": record.get("extra_info", {}).get("question"),
        "critical_failure": record.get("status", {}).get("critical_failure"),
        "reward": reward.get("reward"),
        "score": _get_score_field(record, "score"),
        "format_reward": _get_score_field(record, "format_reward"),
        "accuracy_reward": _get_score_field(record, "accuracy_reward"),
        "n_valid_tool_calls": _get_score_field(record, "n_valid_tool_calls"),
        "extracted_answer": reward.get("extracted_answer"),
        "ground_truth": reward.get("ground_truth"),
        "question_type": record.get("extra_info", {}).get("question_type"),
    })

summary_df = pd.DataFrame(rows)
summary_df


In [ ]:
summary_df[summary_df['file'] == 'insight_doc_mixed-val-trial0-dude_ebda6db2b0501934e41cd6c77516a53d_431f2887699ba54a3538f7fa33b87a66-f8d8cde55bee.json']

In [ ]:
summary_df['subset'].value_counts(dropna=False).sort_index()

In [ ]:
summary_df[summary_df['accuracy_reward'] == 1.0][summary_df['question_type'] == 'not-answerable']

In [ ]:
summary_df[summary_df['accuracy_reward'] == 1.0]['subset'].value_counts(dropna=False).sort_index()

In [ ]:
summary_df['n_valid_tool_calls'].value_counts(dropna=False).sort_index()

In [ ]:
summary_df[summary_df['reward'] == 1.0]

In [ ]:
summary_df[summary_df['reward'] == 1.0]['n_valid_tool_calls'].value_counts(dropna=False).sort_index()

In [ ]:
summary_df[summary_df['reward'] == 1.0][summary_df['n_valid_tool_calls'] > 9]

In [ ]:
OMIT_SYSTEM_PROMPT = False

from IPython.display import Markdown, Pretty, display

def display_restored_conversation(restored_payload):
    image_cnt = 0

    def display_next_image():
        nonlocal image_cnt
        image = restored_payload["multi_modal_data"]["images"][image_cnt]
        if image is None:
            display(Pretty("[image unavailable from reference]"))
        else:
            display(image)
            print("image size:", image.size)
        image_cnt += 1

    display(Markdown(f"`[{restored_payload['record']['agent_name']}]`"))

    for message in restored_payload["messages"]:
        display(Markdown(f"**{message['role'].capitalize()}:**"))

        if OMIT_SYSTEM_PROMPT and message["role"] == "system":
            display(Pretty("[system prompt omitted]"))
            continue

        contents = message["content"] if isinstance(message["content"], list) else [message["content"]]
        for content in contents:
            if isinstance(content, str):
                content = {"type": "text", "text": content}

            if content["type"] == "text":
                if content["text"]:
                    display(Pretty(content["text"]))
            elif content["type"] == "image":
                display_next_image()
            else:
                raise ValueError(f"Unknown content type: {content['type']}")

def display_conversation_by_index(idx):
    path = export_paths[idx]
    record = records[idx]
    restored = restore_conversation_for_visualization(record)

    display(summary_df.loc[idx])
    display(pd.json_normalize(record.get("reward") or {}, sep=".").T)
    display(pd.DataFrame([
        {
            "presented_img_idx": item.get("presented_img_idx"),
            "kind": item.get("kind"),
            "source_original_img_idx": item.get("source_original_img_idx"),
            "parent_presented_img_idx": item.get("parent_presented_img_idx"),
            "bbox_on_original": item.get("bbox_on_original"),
            "display_size": item.get("display_size"),
            "region_description": item.get("region_description"),
            "image_restored": item.get("image") is not None,
        }
        for item in restored["presented_images"]
    ]))
    print("EXPORT_PATH:", path)
    display_restored_conversation(restored)


In [ ]:
summary_df[summary_df['reward'] == 1.0].index

In [ ]:
summary_df[summary_df['reward'] == 1.0][summary_df['subset'] == 'map'].index[-100:]

In [ ]:
summary_df[summary_df['reward'] == 1.0][summary_df['n_valid_tool_calls'] > 5].index[-100:]

In [ ]:
summary_df[summary_df['accuracy_reward'] == 1.0][summary_df['question_type'] == 'not-answerable'].index[-100:]

In [ ]:
# Pick a row index from summary_df and run this cell.
idx = 0
display_conversation_by_index(idx)
print('extracted answer:', summary_df.loc[idx, 'extracted_answer'])
print('ground truth:', summary_df.loc[idx, 'ground_truth'])